In [ ]:
!nvidia-smi

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs detected:", gpus)

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("GPU setup complete." if gpus else "No GPU detected.")

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Breast_Cancer_Project")
MODULE2_DIR = PROJECT_DIR / "module_2"

EXPERIMENT_DIR = MODULE2_DIR / "experiments" / "fine_tune_last10"
for folder in ("models", "results", "figures"):
    (EXPERIMENT_DIR / folder).mkdir(parents=True, exist_ok=True)


for folder in ["models", "results", "figures"]:
    (MODULE2_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Module 2 folder:", MODULE2_DIR)
print("Folders created successfully.")

In [ ]:
# EXPERIMENT OUTPUT PROTECTION
# Existing model and result files must not be overwritten.

from pathlib import Path

_existing_outputs = [
    p
    for folder in ("models", "results", "figures")
    for p in (EXPERIMENT_DIR / folder).rglob("*")
    if p.is_file()
]

_SAFE_WRITE_ALLOWED = len(_existing_outputs) == 0
_SAFE_OUTPUT_DIRECTORY = str(EXPERIMENT_DIR.resolve())
_SAFE_EXECUTED_WRITE_CELLS = set()

print("EXPERIMENT OUTPUT SAFETY")
print("Existing output files:", len(_existing_outputs))

if _SAFE_WRITE_ALLOWED:
    print("New empty output directory detected.")
    print("Single-run writes authorized.")
else:
    print("PROTECTED MODE: Existing results detected.")
    print("Model training and output-writing cells are blocked.")
    print("Use a new experiment directory for a new run.")


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

print("Random seed:", SEED)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Setup complete.")

In [ ]:
from pathlib import Path
import subprocess
import tarfile

DATA_DIR = Path("/content/breakhis")
ARCHIVE_PATH = Path("/content/BreaKHis_v1.tar.gz")

DATA_URL = (
    "https://www.inf.ufpr.br/vri/databases/"
    "BreaKHis_v1.tar.gz"
)

print("Checking BreaKHis dataset...")

images = list(DATA_DIR.rglob("SOB_*.png"))

if len(images) != 7909:
    print("Dataset incomplete. Restoring images...")

    if not ARCHIVE_PATH.is_file() or not tarfile.is_tarfile(ARCHIVE_PATH):
        subprocess.run(
            ["wget", "-O", str(ARCHIVE_PATH), DATA_URL],
            check=True
        )

    if not tarfile.is_tarfile(ARCHIVE_PATH):
        raise RuntimeError("Dataset archive is invalid.")

    DATA_DIR.mkdir(parents=True, exist_ok=True)

    with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
        archive.extractall(DATA_DIR, filter="data")

images = list(DATA_DIR.rglob("SOB_*.png"))

print("\nDATASET VERIFICATION")
print("Dataset location:", DATA_DIR)
print("Images found:", len(images))

if len(images) == 7909:
    print("STATUS: SUCCESS - Dataset ready.")
else:
    raise RuntimeError("Dataset restoration incomplete.")

In [ ]:

from pathlib import Path

# Reuse only an existing complete BreaKHis dataset.
# Do not download, extract, or modify source images.

candidate_dirs = [
    Path("/content/breakhis"),
    MODULE2_DIR / "data",
    PROJECT_DIR / "data" / "breakhis",
]

DATA_DIR = None
image_paths = []

for candidate in candidate_dirs:
    if not candidate.is_dir():
        continue

    found = sorted(candidate.rglob("SOB_*.png"))

    if len(found) == 7909:
        DATA_DIR = candidate
        image_paths = found
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Complete BreaKHis dataset not found. "
        "Expected 7,909 images. "
        "Restore or mount the dataset before training."
    )

print("Existing dataset:", DATA_DIR)
print("Total images found:", len(image_paths))
print("Dataset reuse successful.")


In [ ]:
# SAFETY GUARD - Original clean cell 7
if (
    not globals().get("_SAFE_WRITE_ALLOWED", False)
    or globals().get("_SAFE_OUTPUT_DIRECTORY")
       != str(EXPERIMENT_DIR.resolve())
):
    raise FileExistsError(
        "Experiment output writes are blocked. "
        "Existing results must be preserved. "
        "Use a new experiment directory for a new run."
    )

if 7 in _SAFE_EXECUTED_WRITE_CELLS:
    raise RuntimeError(
        "This output-writing cell was already executed "
        "in the current authorized run."
    )

_SAFE_EXECUTED_WRITE_CELLS.add(7)

import pandas as pd

records = []

for path in image_paths:
    # Example: SOB_B_A-14-22549AB-100-001.png
    parts = path.stem.split("-")
    prefix = parts[0].split("_")

    if len(parts) != 5 or prefix[1] not in {"B", "M"}:
        raise ValueError(f"Unexpected filename: {path.name}")

    label = 0 if prefix[1] == "B" else 1

    records.append({
        "filepath": str(path),
        "filename": path.name,
        "label": label,
        "class_name": "benign" if label == 0 else "malignant",
        "patient_id": f"{parts[1]}-{parts[2]}",
        "magnification": int(parts[3]),
        "subtype": prefix[2]
    })

metadata = pd.DataFrame(records)

# Check that each patient has only one class label.
assert metadata.groupby("patient_id")["label"].nunique().max() == 1

metadata.to_csv(
    EXPERIMENT_DIR / "results" / "image_metadata.csv",
    index=False
)

print("Total images:", len(metadata))
print("Unique patients:", metadata["patient_id"].nunique())

print("\nImages by class:")
print(metadata["class_name"].value_counts())

print("\nImages by magnification:")
print(metadata["magnification"].value_counts().sort_index())

display(metadata.head())

In [ ]:
check = metadata.copy()

# Preserve the class and subtype in a second identifier.
check["sample_group"] = check["filename"].apply(
    lambda name: "-".join(name.removesuffix(".png").split("-")[:3])
)

print("Patient IDs:", check["patient_id"].nunique())
print("Sample groups:", check["sample_group"].nunique())

group_counts = check.groupby("patient_id")["sample_group"].nunique()
shared_ids = group_counts[group_counts > 1].index

print("\nPatient IDs appearing in multiple sample groups:")
if len(shared_ids):
    display(
        check.loc[
            check["patient_id"].isin(shared_ids),
            ["patient_id", "sample_group", "class_name", "subtype"]
        ].drop_duplicates().sort_values("patient_id")
    )
else:
    print("None found.")

In [ ]:
# SAFETY GUARD - Original clean cell 9
if (
    not globals().get("_SAFE_WRITE_ALLOWED", False)
    or globals().get("_SAFE_OUTPUT_DIRECTORY")
       != str(EXPERIMENT_DIR.resolve())
):
    raise FileExistsError(
        "Experiment output writes are blocked. "
        "Existing results must be preserved. "
        "Use a new experiment directory for a new run."
    )

if 9 in _SAFE_EXECUTED_WRITE_CELLS:
    raise RuntimeError(
        "This output-writing cell was already executed "
        "in the current authorized run."
    )

_SAFE_EXECUTED_WRITE_CELLS.add(9)

from sklearn.model_selection import train_test_split

patients = metadata[["patient_id", "label"]].drop_duplicates()

train_patients, remaining_patients = train_test_split(
    patients,
    test_size=0.30,
    stratify=patients["label"],
    random_state=SEED
)

val_patients, test_patients = train_test_split(
    remaining_patients,
    test_size=0.50,
    stratify=remaining_patients["label"],
    random_state=SEED
)

train_df = metadata[
    metadata["patient_id"].isin(train_patients["patient_id"])
].copy()

val_df = metadata[
    metadata["patient_id"].isin(val_patients["patient_id"])
].copy()

test_df = metadata[
    metadata["patient_id"].isin(test_patients["patient_id"])
].copy()

# Confirm there is no patient overlap.
train_ids = set(train_df["patient_id"])
val_ids = set(val_df["patient_id"])
test_ids = set(test_df["patient_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
assert len(train_df) + len(val_df) + len(test_df) == len(metadata)

for name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    df.to_csv(
        EXPERIMENT_DIR / "results" / f"{name}_metadata.csv",
        index=False
    )

    print(f"\n{name.upper()}")
    print("Patient groups:", df["patient_id"].nunique())
    print("Images:", len(df))
    print(df["class_name"].value_counts())

print("\nNo patient overlap. Split files saved.")

In [ ]:
# SAFETY GUARD - Original clean cell 10
if (
    not globals().get("_SAFE_WRITE_ALLOWED", False)
    or globals().get("_SAFE_OUTPUT_DIRECTORY")
       != str(EXPERIMENT_DIR.resolve())
):
    raise FileExistsError(
        "Experiment output writes are blocked. "
        "Existing results must be preserved. "
        "Use a new experiment directory for a new run."
    )

if 10 in _SAFE_EXECUTED_WRITE_CELLS:
    raise RuntimeError(
        "This output-writing cell was already executed "
        "in the current authorized run."
    )

_SAFE_EXECUTED_WRITE_CELLS.add(10)

from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for row, class_name in enumerate(["benign", "malignant"]):
    samples = train_df[
        train_df["class_name"] == class_name
    ].sample(n=4, random_state=SEED)

    for col, (_, sample) in enumerate(samples.iterrows()):
        with Image.open(sample["filepath"]) as image:
            axes[row, col].imshow(image.convert("RGB"))

        axes[row, col].set_title(
            f"{class_name.capitalize()}\n"
            f"Patient: {sample['patient_id']} | "
            f"{sample['magnification']}×"
        )
        axes[row, col].axis("off")

plt.suptitle("BreaKHis: Sample Training Images", fontsize=16)
plt.tight_layout()

plt.savefig(
    EXPERIMENT_DIR / "figures" / "training_image_samples.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.io.decode_png(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)

    # Keep pixel values in the 0–255 range for now.
    # Model-specific preprocessing will be added with the model.
    label = tf.cast(label, tf.float32)
    label = tf.reshape(label, (1,))

    return image, label

def make_dataset(df, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (
            df["filepath"].to_numpy(),
            df["label"].to_numpy(dtype=np.int32)
        )
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

images, labels = next(iter(train_ds))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Pixel range:", float(tf.reduce_min(images)),
      "to", float(tf.reduce_max(images)))
print("Labels: 0 = benign, 1 = malignant")
print("Image pipelines ready.")

In [ ]:
from tensorflow.keras import layers, Model

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.10, seed=SEED + 1),
], name="augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = layers.Input(shape=(*IMAGE_SIZE, 3), name="image")
x = augmentation(inputs)
x = layers.Rescaling(1 / 127.5, offset=-1, name="preprocessing")(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30, seed=SEED)(x)
outputs = layers.Dense(
    1,
    activation="sigmoid",
    name="malignant_score"
)(x)

model = Model(inputs, outputs, name="breast_tissue_classifier")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model.summary()
print("\nModel built successfully.")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].to_numpy()
)

class_weights = {
    int(label): float(weight)
    for label, weight in zip(classes, weights)
}

BEST_MODEL_PATH = MODULE2_DIR / "models" / "mobilenetv2_stage1_best.keras"

# Reuse the original Stage 1 checkpoint.
# Do not retrain or overwrite the original model.
if not BEST_MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Stage 1 checkpoint missing: {BEST_MODEL_PATH}"
    )

print("Stage 1 training skipped.")
print("Existing checkpoint:", BEST_MODEL_PATH)
print("Class weights:", class_weights)


In [ ]:
# Start from the best Stage 1 checkpoint.
model = tf.keras.models.load_model(str(BEST_MODEL_PATH))

base_model = model.get_layer("mobilenetv2_1.00_224")
base_model.trainable = True

# Allow only the last 10 layers to adapt.
for layer in base_model.layers[:-10]:
    layer.trainable = False

# Keep batch-normalization layers frozen.
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Recompile after changing which layers are trainable.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

print(
    "Trainable parameters:",
    sum(int(np.prod(w.shape)) for w in model.trainable_weights)
)
print("Fine-tuning learning rate: 0.00001")
print("Fine-tuning setup complete.")

In [ ]:
# SAFETY GUARD - Original clean cell 15
if (
    not globals().get("_SAFE_WRITE_ALLOWED", False)
    or globals().get("_SAFE_OUTPUT_DIRECTORY")
       != str(EXPERIMENT_DIR.resolve())
):
    raise FileExistsError(
        "Experiment output writes are blocked. "
        "Existing results must be preserved. "
        "Use a new experiment directory for a new run."
    )

if 15 in _SAFE_EXECUTED_WRITE_CELLS:
    raise RuntimeError(
        "This output-writing cell was already executed "
        "in the current authorized run."
    )

_SAFE_EXECUTED_WRITE_CELLS.add(15)

STAGE2_MODEL_PATH = (
    EXPERIMENT_DIR / "models" / "mobilenetv2_stage2_best.keras"
)

callbacks_stage2 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(STAGE2_MODEL_PATH),
        monitor="val_roc_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_roc_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_roc_auc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        str(EXPERIMENT_DIR / "results" / "stage2_training_log.csv")
    )
]

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks_stage2
)

pd.DataFrame(history_stage2.history).to_csv(
    EXPERIMENT_DIR / "results" / "stage2_history.csv",
    index=False
)

print("\nFine-tuning complete.")
print("Best Stage 2 model saved to:", STAGE2_MODEL_PATH)
print(
    "Best Stage 2 validation ROC-AUC:",
    round(max(history_stage2.history["val_roc_auc"]), 4)
)

In [ ]:
# SAFETY GUARD - Original clean cell 16
if (
    not globals().get("_SAFE_WRITE_ALLOWED", False)
    or globals().get("_SAFE_OUTPUT_DIRECTORY")
       != str(EXPERIMENT_DIR.resolve())
):
    raise FileExistsError(
        "Experiment output writes are blocked. "
        "Existing results must be preserved. "
        "Use a new experiment directory for a new run."
    )

if 16 in _SAFE_EXECUTED_WRITE_CELLS:
    raise RuntimeError(
        "This output-writing cell was already executed "
        "in the current authorized run."
    )

_SAFE_EXECUTED_WRITE_CELLS.add(16)

from sklearn.metrics import roc_auc_score, average_precision_score

final_model = tf.keras.models.load_model(
    str(STAGE2_MODEL_PATH)
)

val_scores = final_model.predict(val_ds, verbose=1).ravel()
val_labels = val_df["label"].to_numpy()

assert len(val_scores) == len(val_labels)
assert np.isfinite(val_scores).all()

validation_predictions = val_df.copy()
validation_predictions["malignant_score"] = val_scores

validation_predictions.to_csv(
    EXPERIMENT_DIR / "results" / "validation_predictions.csv",
    index=False
)

print("\nValidation images:", len(val_scores))
print("ROC-AUC:", round(roc_auc_score(val_labels, val_scores), 4))
print(
    "Average precision:",
    round(average_precision_score(val_labels, val_scores), 4)
)
print("Validation predictions saved.")

In [ ]:
# SAFETY GUARD - Original clean cell 17
if (
    not globals().get("_SAFE_WRITE_ALLOWED", False)
    or globals().get("_SAFE_OUTPUT_DIRECTORY")
       != str(EXPERIMENT_DIR.resolve())
):
    raise FileExistsError(
        "Experiment output writes are blocked. "
        "Existing results must be preserved. "
        "Use a new experiment directory for a new run."
    )

if 17 in _SAFE_EXECUTED_WRITE_CELLS:
    raise RuntimeError(
        "This output-writing cell was already executed "
        "in the current authorized run."
    )

_SAFE_EXECUTED_WRITE_CELLS.add(17)

from sklearn.metrics import (
    roc_curve,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import json

fpr, tpr, thresholds = roc_curve(val_labels, val_scores)

# Maximizing TPR - FPR also maximizes balanced accuracy.
valid = np.isfinite(thresholds) & (thresholds >= 0) & (thresholds <= 1)
candidate_indices = np.flatnonzero(valid)
best_index = candidate_indices[np.argmax((tpr - fpr)[valid])]

SELECTED_THRESHOLD = float(thresholds[best_index])

rows = []

for name, threshold in [
    ("Default", 0.5),
    ("Validation-selected", SELECTED_THRESHOLD)
]:
    predictions = (val_scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        val_labels, predictions, labels=[0, 1]
    ).ravel()

    rows.append({
        "threshold_type": name,
        "threshold": threshold,
        "accuracy": accuracy_score(val_labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            val_labels, predictions
        ),
        "malignant_recall": recall_score(
            val_labels, predictions, zero_division=0
        ),
        "benign_recall": tn / (tn + fp),
        "precision": precision_score(
            val_labels, predictions, zero_division=0
        ),
        "f1": f1_score(val_labels, predictions, zero_division=0)
    })

threshold_comparison = pd.DataFrame(rows)

threshold_comparison.to_csv(
    EXPERIMENT_DIR / "results" / "validation_threshold_comparison.csv",
    index=False
)

with open(EXPERIMENT_DIR / "models" / "threshold_config.json", "w") as f:
    json.dump({
        "threshold": SELECTED_THRESHOLD,
        "selection_dataset": "validation",
        "selection_metric": "image_level_balanced_accuracy",
        "positive_class": "malignant",
        "positive_label": 1,
        "decision_rule": "score >= threshold"
    }, f, indent=2)

display(threshold_comparison.round(4))
print("Selected threshold:", round(SELECTED_THRESHOLD, 6))
print("Threshold configuration saved.")

In [ ]:
# Cell 17: Test evaluation disabled.
# This experiment uses training and validation data only.
# Original code is preserved in the notebook backup.
print("Test evaluation skipped.")


In [ ]:
# Cell 18: Test evaluation disabled.
# This experiment uses training and validation data only.
# Original code is preserved in the notebook backup.
print("Test evaluation skipped.")


In [ ]:
# Cell 19: Test evaluation disabled.
# This experiment uses training and validation data only.
# Original code is preserved in the notebook backup.
print("Test evaluation skipped.")


In [ ]:
# SAFETY GUARD - Original clean cell 21
if (
    not globals().get("_SAFE_WRITE_ALLOWED", False)
    or globals().get("_SAFE_OUTPUT_DIRECTORY")
       != str(EXPERIMENT_DIR.resolve())
):
    raise FileExistsError(
        "Experiment output writes are blocked. "
        "Existing results must be preserved. "
        "Use a new experiment directory for a new run."
    )

if 21 in _SAFE_EXECUTED_WRITE_CELLS:
    raise RuntimeError(
        "This output-writing cell was already executed "
        "in the current authorized run."
    )

_SAFE_EXECUTED_WRITE_CELLS.add(21)

# Load the original Stage 1 training history.
# Stage 1 is not retrained in this experiment.
stage1_path = MODULE2_DIR / "results" / "stage1_history.csv"

if not stage1_path.is_file():
    raise FileNotFoundError(
        f"Original Stage 1 history missing: {stage1_path}"
    )

stage1 = pd.read_csv(stage1_path)

print("Original Stage 1 history loaded.")
print("Stage 1 epochs:", len(stage1))
stage2 = pd.DataFrame(history_stage2.history)

combined = pd.concat([stage1, stage2], ignore_index=True)
epochs = np.arange(1, len(combined) + 1)
boundary = len(stage1) + 0.5

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric, title in [
    (axes[0], "loss", "Training and Validation Loss"),
    (axes[1], "roc_auc", "Training and Validation ROC-AUC")
]:
    ax.plot(epochs, combined[metric], label="Training")
    ax.plot(epochs, combined[f"val_{metric}"], label="Validation")
    ax.axvline(
        boundary,
        color="gray",
        linestyle="--",
        label="Fine-tuning begins"
    )
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss" if metric == "loss" else "ROC-AUC")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(
    EXPERIMENT_DIR / "figures" / "training_history.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

print("Training history figure saved.")

In [ ]:
# SAFETY GUARD - Original clean cell 22
if (
    not globals().get("_SAFE_WRITE_ALLOWED", False)
    or globals().get("_SAFE_OUTPUT_DIRECTORY")
       != str(EXPERIMENT_DIR.resolve())
):
    raise FileExistsError(
        "Experiment output writes are blocked. "
        "Existing results must be preserved. "
        "Use a new experiment directory for a new run."
    )

if 22 in _SAFE_EXECUTED_WRITE_CELLS:
    raise RuntimeError(
        "This output-writing cell was already executed "
        "in the current authorized run."
    )

_SAFE_EXECUTED_WRITE_CELLS.add(22)

FINAL_MODEL_PATH = (
    EXPERIMENT_DIR / "models" / "breast_tissue_classifier.keras"
)

final_model.save(str(FINAL_MODEL_PATH))

model_config = {
    "model_file": FINAL_MODEL_PATH.name,
    "architecture": "MobileNetV2",
    "selected_checkpoint": STAGE2_MODEL_PATH.name,
    "image_size": list(IMAGE_SIZE),
    "color_mode": "RGB",
    "input_pixel_range": [0, 255],
    "preprocessing": "Rescaling to [-1, 1] included in model",
    "resize_method": "bilinear",
    "labels": {"0": "benign", "1": "malignant"},
    "threshold": SELECTED_THRESHOLD,
    "decision_rule": "score >= threshold",
    "threshold_selection": "validation image-level balanced accuracy",
    "dataset": "BreaKHis",
    "magnifications": [40, 100, 200, 400],
    "random_seed": SEED,
    "intended_use": "Educational research demonstration",
    "score_note": "Model score is not a calibrated cancer probability",
    "evaluation_split": "validation",
    "validation_images": int(len(validation_predictions)),
    "test_evaluation_performed": False
}

CONFIG_PATH = EXPERIMENT_DIR / "models" / "model_config.json"

with open(CONFIG_PATH, "w") as f:
    json.dump(model_config, f, indent=2)

# Verify that the saved model reproduces the original predictions.
reloaded_model = tf.keras.models.load_model(str(FINAL_MODEL_PATH))

sample_images, _ = next(iter(val_ds))

original_scores = final_model(
    sample_images, training=False
).numpy()

reloaded_scores = reloaded_model(
    sample_images, training=False
).numpy()

np.testing.assert_allclose(
    original_scores,
    reloaded_scores,
    rtol=1e-5,
    atol=1e-6
)

print("Final model saved:", FINAL_MODEL_PATH)
print("Configuration saved:", CONFIG_PATH)
print("Reloaded model predictions match.")

In [ ]:
# Original Module 2 ZIP export disabled.
# This experiment will export only its own results.
# The original export code is preserved in the backup.

print("Original ZIP export skipped.")
print("Experiment results remain in:", EXPERIMENT_DIR)


In [ ]:
# SAFETY GUARD - Original clean cell 24
if (
    not globals().get("_SAFE_WRITE_ALLOWED", False)
    or globals().get("_SAFE_OUTPUT_DIRECTORY")
       != str(EXPERIMENT_DIR.resolve())
):
    raise FileExistsError(
        "Experiment output writes are blocked. "
        "Existing results must be preserved. "
        "Use a new experiment directory for a new run."
    )

if 24 in _SAFE_EXECUTED_WRITE_CELLS:
    raise RuntimeError(
        "This output-writing cell was already executed "
        "in the current authorized run."
    )

_SAFE_EXECUTED_WRITE_CELLS.add(24)

# STEP 79 - TRAINING HISTORY COMPARISON
# Uses saved CSV files only. No model training.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Breast_Cancer_Project"
)

MODULE2_DIR = PROJECT_DIR / "module_2"
EXPERIMENT_DIR = (
    MODULE2_DIR / "experiments" / "fine_tune_last10"
)

original_path = (
    MODULE2_DIR / "results" / "stage2_history.csv"
)

experimental_path = (
    EXPERIMENT_DIR / "results" / "stage2_history.csv"
)

figure_path = (
    EXPERIMENT_DIR / "figures" / "training_history.png"
)

print("STEP 79 - TRAINING HISTORY COMPARISON\n")

print("Original history found:", original_path.is_file())
print("Experimental history found:", experimental_path.is_file())

if not original_path.is_file() or not experimental_path.is_file():
    raise FileNotFoundError(
        "One or both saved training histories are missing."
    )

original = pd.read_csv(original_path)
experimental = pd.read_csv(experimental_path)

required_columns = [
    "roc_auc",
    "val_roc_auc",
    "loss",
    "val_loss"
]

for name, df in [
    ("Original", original),
    ("Experimental", experimental)
]:
    missing = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{name} history missing columns: {missing}"
        )

    if df.empty or not np.isfinite(
        df[required_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f"{name} history is empty or contains invalid metrics."
        )

print("\nOriginal Stage 2 epochs:", len(original))
print("Experimental Stage 2 epochs:", len(experimental))

original_best = original["val_roc_auc"].max()
experimental_best = experimental["val_roc_auc"].max()

print("\nOriginal best validation ROC-AUC:",
      round(original_best, 4))

print("Experimental best validation ROC-AUC:",
      round(experimental_best, 4))

print("Difference:",
      round(experimental_best - original_best, 4))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ("roc_auc", "Training ROC-AUC"),
    ("val_roc_auc", "Validation ROC-AUC"),
    ("loss", "Training Loss"),
    ("val_loss", "Validation Loss")
]

for ax, (metric, title) in zip(axes.flat, metrics):
    ax.plot(
        range(1, len(original) + 1),
        original[metric],
        marker="o",
        label="Original: Last 30 layers"
    )

    ax.plot(
        range(1, len(experimental) + 1),
        experimental[metric],
        marker="s",
        label="Experiment: Last 10 layers"
    )

    ax.set_title(title)
    ax.set_xlabel("Stage 2 Epoch")
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle(
    "MobileNetV2 Fine-Tuning Comparison\n"
    "Original Last-30 vs Experimental Last-10",
    fontsize=15
)

plt.tight_layout()

if figure_path.exists():
    print("\nFigure already exists; not overwriting:",
          figure_path)
else:
    plt.savefig(
        figure_path,
        dpi=200,
        bbox_inches="tight"
    )
    print("\nFigure saved:", figure_path)

plt.show()

print("\nSTEP 79 COMPLETE")
print("No models retrained.")
print("No test data evaluated.")

In [ ]:
# SAFETY GUARD - Original clean cell 25
if (
    not globals().get("_SAFE_WRITE_ALLOWED", False)
    or globals().get("_SAFE_OUTPUT_DIRECTORY")
       != str(EXPERIMENT_DIR.resolve())
):
    raise FileExistsError(
        "Experiment output writes are blocked. "
        "Existing results must be preserved. "
        "Use a new experiment directory for a new run."
    )

if 25 in _SAFE_EXECUTED_WRITE_CELLS:
    raise RuntimeError(
        "This output-writing cell was already executed "
        "in the current authorized run."
    )

_SAFE_EXECUTED_WRITE_CELLS.add(25)

# STEP 80 - CREATE EXPERIMENT COMPARISON REPORT
# Uses saved files only. No training or test evaluation.

from pathlib import Path
import pandas as pd
import json
from sklearn.metrics import roc_auc_score, average_precision_score

PROJECT_DIR = Path("/content/drive/MyDrive/Breast_Cancer_Project")
MODULE2_DIR = PROJECT_DIR / "module_2"
EXPERIMENT_DIR = MODULE2_DIR / "experiments" / "fine_tune_last10"

original_path = MODULE2_DIR / "results" / "stage2_history.csv"
experiment_path = EXPERIMENT_DIR / "results" / "stage2_history.csv"
predictions_path = EXPERIMENT_DIR / "results" / "validation_predictions.csv"
comparison_path = EXPERIMENT_DIR / "results" / "validation_threshold_comparison.csv"
config_path = EXPERIMENT_DIR / "models" / "model_config.json"

report_path = EXPERIMENT_DIR / "results" / "fine_tuning_comparison_report.md"

required = [
    original_path,
    experiment_path,
    predictions_path,
    comparison_path,
    config_path
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required file: {path}")

original = pd.read_csv(original_path)
experiment = pd.read_csv(experiment_path)
predictions = pd.read_csv(predictions_path)
comparison = pd.read_csv(comparison_path)

with open(config_path, encoding="utf-8") as f:
    config = json.load(f)

original_best = float(original["val_roc_auc"].max())
experiment_best = float(experiment["val_roc_auc"].max())

original_epoch = int(original["val_roc_auc"].idxmax() + 1)
experiment_epoch = int(experiment["val_roc_auc"].idxmax() + 1)

val_auc = roc_auc_score(
    predictions["label"],
    predictions["malignant_score"]
)

val_ap = average_precision_score(
    predictions["label"],
    predictions["malignant_score"]
)

selected = comparison.loc[
    comparison["threshold_type"] == "Validation-selected"
].iloc[0]

if len(predictions) != 1120:
    raise RuntimeError("Validation prediction count mismatch.")

if config.get("test_evaluation_performed") is not False:
    raise RuntimeError("Unexpected test evaluation configuration.")

report = f"""# MobileNetV2 Fine-Tuning Experiment Report

## 1. Objective

Evaluate whether reducing the number of trainable MobileNetV2
backbone layers from the last 30 to the last 10 improves validation
performance for BreaKHis breast histopathology image classification.

## 2. Dataset

- Dataset: BreaKHis
- Total images: 7,909
- Unique extracted patient IDs: 81
- Training: 5,303 images, 56 patient groups
- Validation: 1,120 images, 12 patient groups
- Held-out test: 1,486 images, 13 patient groups
- Classes: Benign and malignant
- Input size: 224 x 224 RGB
- Patient-group overlap: None detected

## 3. Experimental Method

Both configurations use MobileNetV2 and the original Stage 1
checkpoint as the starting point.

The experiment fine-tunes the last 10 backbone layers instead of
the original last 30, with BatchNormalization layers frozen.

- Learning rate: 0.00001
- Maximum Stage 2 epochs: 20
- Model selection: Validation ROC-AUC
- Early stopping patience: 5
- Class weighting: Applied during training

The original Stage 1 model was reused without retraining.

## 4. Training Results

| Metric | Last 30 layers | Last 10 layers |
|---|---:|---:|
| Stage 2 epochs | {len(original)} | {len(experiment)} |
| Best epoch | {original_epoch} | {experiment_epoch} |
| Best validation ROC-AUC | {original_best:.4f} | {experiment_best:.4f} |

Difference in best validation ROC-AUC (last 10 minus last 30):
{experiment_best-original_best:+.4f}

## 5. Experimental Model Validation

Results from the saved last-10-layer checkpoint:

- Validation images: {len(predictions)}
- ROC-AUC: {val_auc:.4f}
- Average precision: {val_ap:.4f}
- Validation-selected threshold: {float(selected["threshold"]):.6f}
- Accuracy: {float(selected["accuracy"]):.2%}
- Balanced accuracy: {float(selected["balanced_accuracy"]):.2%}
- Malignant recall: {float(selected["malignant_recall"]):.2%}
- Benign recall: {float(selected["benign_recall"]):.2%}
- Precision: {float(selected["precision"]):.2%}
- F1-score: {float(selected["f1"]):.2%}

The classification threshold was selected using validation data
to maximize image-level balanced accuracy.

## 6. Interpretation

The last-30-layer configuration achieved a higher best validation
ROC-AUC than the last-10-layer experiment.

The last-10-layer model achieved relatively high malignant recall
at its selected threshold, but benign recall remained low, indicating
many benign images were classified as malignant.

The results support retaining the original last-30-layer model as
the stronger candidate on this validation split.

## 7. Limitations

- The comparison uses a single patient-group split.
- Validation data was used for model and threshold selection.
- The experiments used different numbers of completed epochs.
- Training randomness may affect the comparison.
- The experimental model was not evaluated on the test set.
- These image-level results do not establish clinical performance.
- The model is intended for educational research only.

## 8. Conclusion

Fine-tuning the last 10 MobileNetV2 backbone layers did not improve
validation ROC-AUC compared with the original last-30-layer approach.

The comparison is exploratory and does not establish statistical
significance or generalization to independent clinical data.
"""

if report_path.exists():
    print("Report already exists; not overwritten:")
else:
    report_path.write_text(report, encoding="utf-8")
    print("Comparison report created successfully.")

print("\nSTEP 80 - REPORT SUMMARY")
print("Original best validation ROC-AUC:", round(original_best, 4))
print("Experimental best validation ROC-AUC:", round(experiment_best, 4))
print("Difference:", round(experiment_best - original_best, 4))
print("Experimental independent validation ROC-AUC:", round(val_auc, 4))
print("Report saved at:", report_path)
print("\nNo models retrained.")
print("No test data evaluated.")

In [ ]:
# STEP 81 - FINAL EXPERIMENT AUDIT
# Read-only: no training, file changes, or test evaluation.

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score
)

project = Path("/content/drive/MyDrive/Breast_Cancer_Project")
module = project / "module_2"
experiment = module / "experiments" / "fine_tune_last10"

required_files = {
    "Experimental notebook": experiment / "02_fine_tune_last10.ipynb",
    "Stage 2 checkpoint": experiment / "models/mobilenetv2_stage2_best.keras",
    "Final model": experiment / "models/breast_tissue_classifier.keras",
    "Model configuration": experiment / "models/model_config.json",
    "Threshold configuration": experiment / "models/threshold_config.json",
    "Training history": experiment / "results/stage2_history.csv",
    "Training log": experiment / "results/stage2_training_log.csv",
    "Validation predictions": experiment / "results/validation_predictions.csv",
    "Threshold comparison": experiment / "results/validation_threshold_comparison.csv",
    "Comparison report": experiment / "results/fine_tuning_comparison_report.md",
    "Training history figure": experiment / "figures/training_history.png",
    "Sample training figure": experiment / "figures/training_image_samples.png",
}

print("STEP 81 - FINAL EXPERIMENT AUDIT\n")

print("1. EXPERIMENTAL FILES")
for name, path in required_files.items():
    print(f"{'PASS' if path.is_file() else 'MISSING'}: {name}")

if not all(path.is_file() for path in required_files.values()):
    print("\nSTATUS: INCOMPLETE - Missing experimental files.")
    raise RuntimeError("Review missing files before continuing.")

print("\n2. ORIGINAL MODEL SEPARATION")

original_stage1 = module / "models/mobilenetv2_stage1_best.keras"
original_stage2 = module / "models/mobilenetv2_stage2_best.keras"
experimental_stage2 = required_files["Stage 2 checkpoint"]

checks = {
    "Original Stage 1 exists": original_stage1.is_file(),
    "Original Stage 2 exists": original_stage2.is_file(),
    "Experimental checkpoint uses separate path":
        original_stage2.resolve() != experimental_stage2.resolve(),
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

print("\n3. RESULT CONSISTENCY")

original = pd.read_csv(module / "results/stage2_history.csv")
new = pd.read_csv(required_files["Training history"])
predictions = pd.read_csv(required_files["Validation predictions"])
thresholds = pd.read_csv(required_files["Threshold comparison"])

with open(required_files["Model configuration"]) as f:
    config = json.load(f)

with open(required_files["Threshold configuration"]) as f:
    threshold_config = json.load(f)

original_auc = float(original["val_roc_auc"].max())
new_auc = float(new["val_roc_auc"].max())

y_true = predictions["label"].to_numpy()
scores = predictions["malignant_score"].to_numpy()

validation_auc = roc_auc_score(y_true, scores)
validation_ap = average_precision_score(y_true, scores)

selected_threshold = float(threshold_config["threshold"])
predicted = (scores >= selected_threshold).astype(int)

calculated_balanced_accuracy = balanced_accuracy_score(
    y_true, predicted
)

selected_rows = thresholds.loc[
    thresholds["threshold_type"] == "Validation-selected"
]

if len(selected_rows) != 1:
    raise RuntimeError("Expected exactly one selected-threshold row.")

selected_row = selected_rows.iloc[0]

result_checks = {
    "Validation contains 1120 images":
        len(predictions) == 1120
        and config.get("validation_images") == 1120,

    "Validation scores are valid":
        np.isfinite(scores).all()
        and np.all((scores >= 0) & (scores <= 1)),

    "Model and threshold configurations agree":
        np.isclose(
            float(config["threshold"]),
            selected_threshold,
            atol=1e-8,
            rtol=0
        ),

    "Threshold comparison agrees with predictions":
        np.isclose(
            calculated_balanced_accuracy,
            float(selected_row["balanced_accuracy"]),
            atol=1e-8,
            rtol=0
        ),

    "Saved checkpoint ROC-AUC agrees with training":
        abs(validation_auc - new_auc) < 0.005,

    "Original model has higher validation ROC-AUC":
        original_auc > new_auc,

    "Test evaluation marked not performed":
        config.get("test_evaluation_performed") is False,
}

for name, passed in result_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

print("\n4. FINAL RESULTS")
print("Original best validation ROC-AUC:", round(original_auc, 4))
print("Experimental best validation ROC-AUC:", round(new_auc, 4))
print("Experimental validation ROC-AUC:", round(validation_auc, 4))
print("Experimental average precision:", round(validation_ap, 4))
print("Selected threshold:", round(selected_threshold, 6))
print("Selected balanced accuracy:", round(calculated_balanced_accuracy, 4))
print("Difference (last 10 - last 30):", round(new_auc - original_auc, 4))

all_passed = all(checks.values()) and all(result_checks.values())

print("\nFINAL AUDIT STATUS:")
print("PASS" if all_passed else "REVIEW REQUIRED")
print("No training performed.")
print("No test data evaluated.")
print("No files modified.")

In [ ]:
# SAFETY GUARD - Original clean cell 27
if (
    not globals().get("_SAFE_WRITE_ALLOWED", False)
    or globals().get("_SAFE_OUTPUT_DIRECTORY")
       != str(EXPERIMENT_DIR.resolve())
):
    raise FileExistsError(
        "Experiment output writes are blocked. "
        "Existing results must be preserved. "
        "Use a new experiment directory for a new run."
    )

if 27 in _SAFE_EXECUTED_WRITE_CELLS:
    raise RuntimeError(
        "This output-writing cell was already executed "
        "in the current authorized run."
    )

_SAFE_EXECUTED_WRITE_CELLS.add(27)

# STEP 82 - CREATE EXPERIMENT README
# Reads saved results only. Does not modify models.

from pathlib import Path
import json
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

project = Path("/content/drive/MyDrive/Breast_Cancer_Project")
module = project / "module_2"
experiment = module / "experiments" / "fine_tune_last10"

original_history_path = module / "results/stage2_history.csv"
new_history_path = experiment / "results/stage2_history.csv"
predictions_path = experiment / "results/validation_predictions.csv"
threshold_path = experiment / "models/threshold_config.json"
comparison_path = experiment / "results/validation_threshold_comparison.csv"
report_path = experiment / "results/fine_tuning_comparison_report.md"

readme_path = experiment / "README.md"

required = [
    original_history_path,
    new_history_path,
    predictions_path,
    threshold_path,
    comparison_path,
    report_path
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required file: {path}")

original = pd.read_csv(original_history_path)
new = pd.read_csv(new_history_path)
predictions = pd.read_csv(predictions_path)
comparison = pd.read_csv(comparison_path)

with open(threshold_path, encoding="utf-8") as f:
    threshold_config = json.load(f)

original_auc = float(original["val_roc_auc"].max())
new_auc = float(new["val_roc_auc"].max())

original_epoch = int(original["val_roc_auc"].idxmax() + 1)
new_epoch = int(new["val_roc_auc"].idxmax() + 1)

validation_auc = roc_auc_score(
    predictions["label"],
    predictions["malignant_score"]
)

average_precision = average_precision_score(
    predictions["label"],
    predictions["malignant_score"]
)

selected_threshold = float(threshold_config["threshold"])

selected_rows = comparison.loc[
    comparison["threshold_type"] == "Validation-selected"
]

if len(predictions) != 1120 or len(selected_rows) != 1:
    raise RuntimeError("Unexpected validation results.")

selected = selected_rows.iloc[0]

readme = f"""# MobileNetV2 Last-10-Layer Fine-Tuning Experiment

## Overview

This experiment investigates whether fine-tuning fewer MobileNetV2
backbone layers improves breast histopathology image classification
on the BreaKHis dataset.

The original configuration fine-tuned the last 30 backbone layers.
This experiment fine-tuned the last 10 backbone layers.

Both configurations use the original trained Stage 1 checkpoint
as their starting point.

**Purpose:** Educational machine learning research.
**Not intended for clinical diagnosis or medical decision-making.**

## Dataset

- Dataset: BreaKHis
- Total images: 7,909
- Extracted patient identifiers: 81
- Training: 5,303 images from 56 patient groups
- Validation: 1,120 images from 12 patient groups
- Original test partition: 1,486 images from 13 patient groups
- Classes: Benign (0), Malignant (1)
- Input: 224 x 224 RGB images

The split was created using extracted patient identifiers.
No patient-group overlap was detected across partitions.

The original test partition was not evaluated in this experiment.

## Model Configuration

- Architecture: MobileNetV2
- Starting weights: Original best Stage 1 checkpoint
- Fine-tuning: Last 10 backbone layers
- Earlier backbone layers: Frozen
- BatchNormalization: Frozen
- Classification head: Trainable
- Recorded trainable parameters: 726,721
- Optimizer: Adam
- Initial fine-tuning learning rate: 0.00001
- Maximum Stage 2 epochs: 20
- Early stopping patience: 5
- Checkpoint selection: Best validation ROC-AUC
- Class weighting: Applied
- Random seed: 42

Input rescaling from [0, 255] to approximately [-1, 1]
is included in the saved model.

## Fine-Tuning Comparison

| Metric | Last 30 Layers | Last 10 Layers |
|---|---:|---:|
| Stage 2 epochs completed | {len(original)} | {len(new)} |
| Best epoch | {original_epoch} | {new_epoch} |
| Best validation ROC-AUC | {original_auc:.4f} | {new_auc:.4f} |

**Difference (last 10 minus last 30):**
{new_auc-original_auc:+.4f} ROC-AUC.

## Experimental Model Validation

The best last-10-layer checkpoint was reloaded and evaluated
on the validation partition.

- Validation images: {len(predictions)}
- Validation ROC-AUC: {validation_auc:.4f}
- Average precision: {average_precision:.4f}
- Selected classification threshold: {selected_threshold:.6f}
- Accuracy: {float(selected["accuracy"]):.2%}
- Balanced accuracy: {float(selected["balanced_accuracy"]):.2%}
- Malignant recall: {float(selected["malignant_recall"]):.2%}
- Benign recall: {float(selected["benign_recall"]):.2%}
- Precision: {float(selected["precision"]):.2%}
- F1-score: {float(selected["f1"]):.2%}

The threshold was selected on the validation set using
image-level balanced accuracy.

## Findings

The original last-30-layer model achieved higher validation
ROC-AUC than the experimental last-10-layer model.

Reducing the number of trainable backbone layers did not
improve validation discrimination in this experiment.

The experimental model achieved high malignant recall at
its validation-selected threshold, but benign recall remained low.

This indicates a substantial false-positive limitation.

## Limitations

- Only one patient-group split was investigated.
- Results are based on image-level validation metrics.
- Model selection and threshold selection used validation data.
- Different training durations and stochastic training effects
  limit causal interpretation of the comparison.
- No additional test-set evaluation was performed in this experiment.
- The broader project's original test set had previously been examined.
- No statistical significance or external validation was established.
- These results do not establish clinical reliability.

## Experiment Files

### Notebook
- `02_fine_tune_last10.ipynb`

### Models
- `models/mobilenetv2_stage2_best.keras`
- `models/breast_tissue_classifier.keras`
- `models/model_config.json`
- `models/threshold_config.json`

### Results
- `results/stage2_history.csv`
- `results/stage2_training_log.csv`
- `results/validation_predictions.csv`
- `results/validation_threshold_comparison.csv`
- `results/fine_tuning_comparison_report.md`

### Figures
- `figures/training_history.png`
- `figures/training_image_samples.png`

## Reproducibility Notes

The experiment requires the original trained Stage 1 checkpoint,
the BreaKHis dataset, and the project dependencies.

The notebook documents the patient-level split,
preprocessing, training configuration, and validation procedure.

To repeat training, use a separate experiment directory so that
existing checkpoints and result files are not overwritten.

Reproducing the exact numerical results may depend on
software versions and nondeterministic GPU operations.

The original BreaKHis images are not distributed with
this experiment's result files.

## Conclusion

The last-30-layer configuration remains the preferred
candidate based on validation ROC-AUC.

The last-10-layer configuration is retained as a documented,
reproducible experimental comparison.

Further evaluation on independent patient data would be
needed before making broader generalization claims.
"""

print("STEP 82 - EXPERIMENT DOCUMENTATION\n")

if readme_path.exists():
    print("README already exists. Not overwritten.")
else:
    readme_path.write_text(readme, encoding="utf-8")
    print("README created successfully.")

print("\nREADME location:", readme_path)
print("README exists:", readme_path.is_file())
print("Original validation ROC-AUC:", round(original_auc, 4))
print("Experimental validation ROC-AUC:", round(validation_auc, 4))
print("Selected threshold:", round(selected_threshold, 6))
print("\nNo training performed.")
print("No test data evaluated.")
print("No model files modified.")